In [1]:
# CELL 1
# Install all tools your AI needs
# Tap the ▶️ button to run each cell

!pip install torch numpy transformers datasets

import torch
import torch.nn as nn
import numpy as np

# Check if GPU is available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"✅ Everything installed!")
print(f"   Device: {device}")
print(f"   If it says 'cpu' go to Runtime → Change Runtime Type → T4 GPU")

✅ Everything installed!
   Device: cuda
   If it says 'cpu' go to Runtime → Change Runtime Type → T4 GPU


In [2]:
# CELL 2
# Kaggle saves your work automatically!
# No Google Drive setup needed at all
# Your files stay safe at /kaggle/working/

import os

# This is your Kaggle save folder
# Everything here persists between sessions
SAVE = '/kaggle/working/MyGameAI'
os.makedirs(f'{SAVE}/model',       exist_ok=True)
os.makedirs(f'{SAVE}/data',        exist_ok=True)
os.makedirs(f'{SAVE}/checkpoints', exist_ok=True)

print("✅ Save folders ready!")
print(f"   Location: {SAVE}")
print("   Kaggle keeps your files automatically!")
print("   No Google Drive setup needed!")
print()
print("💡 IMPORTANT KAGGLE SETTINGS:")
print("   On the right side panel in Kaggle set these:")
print("   • Accelerator → GPU T4 x2 (free!)")
print("   • Persistence → Files (keeps your files!)")
print("   • Internet → ON (needed to install packages)")

✅ Save folders ready!
   Location: /kaggle/working/MyGameAI
   Kaggle keeps your files automatically!
   No Google Drive setup needed!

💡 IMPORTANT KAGGLE SETTINGS:
   On the right side panel in Kaggle set these:
   • Accelerator → GPU T4 x2 (free!)
   • Persistence → Files (keeps your files!)
   • Internet → ON (needed to install packages)


In [3]:
# CELL 3
# THIS IS YOUR AI BRAIN
# Read every comment carefully — it explains what each part does

import torch
import torch.nn as nn
import math

# ════════════════════════════════════════
# PART A — TOKENIZER
# Converts words into numbers
# AI cannot read words — only numbers
# ════════════════════════════════════════

class SimpleTokenizer:
    """
    Turns text into numbers and back.

    Example:
    "make a game" → [45, 2, 89]
    [45, 2, 89]   → "make a game"
    """

    def __init__(self):
        self.word_to_id = {"<PAD>": 0, "<START>": 1, "<END>": 2, "<UNK>": 3}
        self.id_to_word = {0: "<PAD>", 1: "<START>", 2: "<END>", 3: "<UNK>"}
        self.next_id = 4

    def add_text(self, text):
        """Learn new words from text."""
        for word in text.lower().split():
            if word not in self.word_to_id:
                self.word_to_id[word] = self.next_id
                self.id_to_word[self.next_id] = word
                self.next_id += 1

    def encode(self, text, max_length=128):
        """Text → numbers."""
        words = text.lower().split()
        ids = [self.word_to_id.get(w, 3) for w in words]  # 3 = unknown word
        ids = ids[:max_length]  # cut if too long
        ids += [0] * (max_length - len(ids))  # pad if too short
        return ids

    def decode(self, ids):
        """Numbers → text."""
        words = []
        for i in ids:
            if i == 2:  # END token
                break
            if i not in (0, 1):  # skip PAD and START
                words.append(self.id_to_word.get(i, "<UNK>"))
        return " ".join(words)

    def vocab_size(self):
        return self.next_id

    def save(self, path):
        import json
        with open(path, 'w') as f:
            json.dump({'word_to_id': self.word_to_id,
                      'id_to_word': {str(k): v for k,v in self.id_to_word.items()},
                      'next_id': self.next_id}, f)
        print(f"Tokenizer saved: {path}")

    def load(self, path):
        import json
        with open(path, 'r') as f:
            data = json.load(f)
        self.word_to_id = data['word_to_id']
        self.id_to_word = {int(k): v for k,v in data['id_to_word'].items()}
        self.next_id = data['next_id']
        print(f"Tokenizer loaded: {path}")

tokenizer = SimpleTokenizer()
print("✅ Tokenizer created!")


# ════════════════════════════════════════
# PART B — ATTENTION MECHANISM
# Helps AI focus on important words
# Like how you focus on key words when reading
# ════════════════════════════════════════

class SelfAttention(nn.Module):
    """
    Attention lets the AI focus on
    the most important words when generating an answer.

    Example:
    Question: "How do I make a JUMP in Godot?"
    AI focuses most on: JUMP and Godot
    Less on: How, do, I, make, a, in
    """

    def __init__(self, embed_dim, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        # These learn what to focus on
        self.query  = nn.Linear(embed_dim, embed_dim)
        self.key    = nn.Linear(embed_dim, embed_dim)
        self.value  = nn.Linear(embed_dim, embed_dim)
        self.output = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        B, T, C = x.shape  # batch, sequence length, channels

        # Split into multiple heads
        def split_heads(tensor):
            return tensor.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        Q = split_heads(self.query(x))
        K = split_heads(self.key(x))
        V = split_heads(self.value(x))

        # Calculate attention scores
        scale = math.sqrt(self.head_dim)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / scale

        # Mask future tokens (AI should not peek at future words)
        mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
        scores = scores.masked_fill(mask, float('-inf'))

        # Convert to probabilities
        attention = torch.softmax(scores, dim=-1)

        # Apply attention to values
        out = torch.matmul(attention, V)
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.output(out)


# ════════════════════════════════════════
# PART C — TRANSFORMER BLOCK
# One thinking layer of your AI
# Your AI has multiple of these stacked
# ════════════════════════════════════════

class TransformerBlock(nn.Module):
    """
    One layer of thinking.
    Your AI stacks multiple of these.
    More layers = smarter AI (but slower to train)
    """

    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()

        # Attention — focuses on important words
        self.attention = SelfAttention(embed_dim, num_heads)

        # Feed forward — processes the information
        self.feed_forward = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.GELU(),                      # activation function
            nn.Linear(ff_dim, embed_dim),
            nn.Dropout(dropout)
        )

        # Normalization — keeps numbers stable during training
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # Attention with residual connection
        x = x + self.dropout(self.attention(self.norm1(x)))
        # Feed forward with residual connection
        x = x + self.dropout(self.feed_forward(self.norm2(x)))
        return x


# ════════════════════════════════════════
# PART D — YOUR COMPLETE AI MODEL
# This is the full brain
# ════════════════════════════════════════

class YourGameAI(nn.Module):
    """
    YOUR COMPLETE AI MODEL

    This is what you built from scratch.
    It takes a question as input
    and generates a game development answer.

    You can adjust these settings:
    vocab_size  = how many words it knows
    embed_dim   = how much memory per word (bigger = smarter)
    num_layers  = how many thinking layers (more = smarter but slower)
    num_heads   = how many attention heads
    max_length  = longest text it can handle
    """

    def __init__(
        self,
        vocab_size,
        embed_dim=512,    # ← increase to 512 for smarter AI
        num_layers=8,     # ← increase to 8 for smarter AI
        num_heads=8,
        ff_dim=512,
        max_length=512,
        dropout=0.1
    ):
        super().__init__()
        self.max_length = max_length

        # Word embeddings — converts word IDs to vectors
        self.word_embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        # Position embeddings — tells AI WHERE each word is
        self.pos_embed = nn.Embedding(max_length, embed_dim)

        # Dropout for regularization
        self.dropout = nn.Dropout(dropout)

        # Stack of transformer blocks (the thinking layers)
        self.layers = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, ff_dim, dropout)
            for _ in range(num_layers)
        ])

        # Final normalization
        self.norm = nn.LayerNorm(embed_dim)

        # Output head — converts back to word probabilities
        self.output_head = nn.Linear(embed_dim, vocab_size, bias=False)

        # Initialize weights properly
        self._init_weights()

        # Count parameters
        params = sum(p.numel() for p in self.parameters())
        print(f"   Your AI has {params:,} parameters")

    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, token_ids):
        B, T = token_ids.shape

        # Get word embeddings
        word_emb = self.word_embed(token_ids)

        # Get position embeddings
        positions = torch.arange(T, device=token_ids.device).unsqueeze(0)
        pos_emb = self.pos_embed(positions)

        # Combine word + position information
        x = self.dropout(word_emb + pos_emb)

        # Pass through all thinking layers
        for layer in self.layers:
            x = layer(x)

        # Final normalization
        x = self.norm(x)

        # Get word probabilities
        logits = self.output_head(x)
        return logits

    def generate(self, tokenizer, prompt, max_new_tokens=200, temperature=0.9, repetition_penalty=1.3):
        """
        Generate an answer given a prompt.
        
        temperature: lower = more focused, higher = more creative
        repetition_penalty: above 1.0 punishes repeated words
                           1.3 is a good starting value
                           higher = less repetition but less fluent
        """
        self.eval()
        ids = tokenizer.encode(prompt, max_length=self.max_length)
        input_ids = torch.tensor([ids], dtype=torch.long, device=next(self.parameters()).device)

        generated = []
        with torch.no_grad():
            for _ in range(max_new_tokens):
                # Only use last max_length tokens
                current = input_ids[:, -self.max_length:]
                logits = self(current)

                # Get next word probabilities
                next_logits = logits[:, -1, :] / temperature

                # REPETITION PENALTY
                # Reduce probability of words already generated
                # This stops "small small small" loops!
                if repetition_penalty != 1.0:
                    for token_id in set(input_ids[0].tolist() + generated):
                        if next_logits[0, token_id] > 0:
                            next_logits[0, token_id] /= repetition_penalty
                        else:
                            next_logits[0, token_id] *= repetition_penalty

                probs = torch.softmax(next_logits, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)

                # Stop if END token
                if next_token.item() == 2:
                    break

                generated.append(next_token.item())
                input_ids = torch.cat([input_ids, next_token], dim=1)

        return tokenizer.decode(generated)

print("✅ All AI classes defined!")
print("   Ready to create your model in the next cell")

✅ Tokenizer created!
✅ All AI classes defined!
   Ready to create your model in the next cell


In [4]:
# CELL 4
# CREATE YOUR AI MODEL
# This actually builds the brain

# You will add vocabulary from training data in Phase 3
# For now create with small vocab — will grow
tokenizer = SimpleTokenizer()

# Add basic game dev words so tokenizer knows them
basic_words = """
extends node scene player enemy coin jump move speed gravity
health score level game godot gdscript characterbody2d
rigidbody2d area2d collisionshape2d sprite2d animatedsprite2d
tilemap canvaslayer label button input velocity position
signal func var const export ready process physics
android mobile touch screen apk export build
platformer shooter puzzle rpg topdown runner
design mechanic difficulty balance feel polish
audio sound music sfx animation sprite pixel
# BASIC GAME DEV WORDS
# Core beginner-friendly Godot & game development terms
extends class_name node scene player enemy npc boss
health mana stamina damage speed score level xp
checkpoint respawn save load pause menu ui hud

# GODOT 4 CORE NODES
# Most commonly used nodes in 2D games
Node Node2D CharacterBody2D StaticBody2D RigidBody2D
Area2D CollisionShape2D CollisionPolygon2D
Sprite2D AnimatedSprite2D Camera2D TileMap
CanvasLayer Control Label Button TextureButton
ProgressBar Timer AudioStreamPlayer2D

# GDSCRIPT FUNDAMENTALS
# Basic scripting structure
func var const enum static
if elif else match
for while break continue pass return
true false null
await yield
signal

# DATA TYPES
# Common variable types
int float bool String Array Dictionary
Vector2 Vector3 Color Transform2D

# INPUT & MOVEMENT
# Player controls and interaction
Input InputEvent InputMap
move_and_slide move_and_collide
velocity direction delta
jump gravity acceleration friction

# GAME GENRES
# Common 2D game types
platformer shooter rpg puzzle survival
arcade idle clicker adventure horror

# MOBILE & ANDROID
# Export and platform keywords
android mobile touch screen apk export
portrait landscape build debug release

# GAME DESIGN TERMS
# Basic mechanics and concepts
mechanic difficulty balance combo cooldown
upgrade skill inventory quest dialogue
animation sprite tileset collision hitbox
camera shake particles sfx music audio

# GODOT 4 – 2D MOBILE FOCUSED KEYWORDS
# Optimized for Android + Touch Controls

# CORE 2D NODES
Node Node2D CharacterBody2D StaticBody2D RigidBody2D
Area2D CollisionShape2D CollisionPolygon2D
Sprite2D AnimatedSprite2D AnimationPlayer
Camera2D TileMap TileSet ParallaxBackground ParallaxLayer
GPUParticles2D CPUParticles2D VisibleOnScreenNotifier2D

# UI FOR MOBILE
Control CanvasLayer Label Button TextureButton
TouchScreenButton VirtualKeyboard LineEdit
ProgressBar TextureProgressBar Panel VBoxContainer HBoxContainer
MarginContainer CenterContainer

# MOVEMENT & PHYSICS
velocity direction speed gravity jump_force
move_and_slide move_and_collide
is_on_floor is_on_wall
acceleration friction knockback dash double_jump

# INPUT SYSTEM (MOBILE)
Input InputEvent InputEventScreenTouch
InputEventScreenDrag InputMap
get_vector get_axis
touch swipe tap hold gesture

# EXPORT & ANDROID BUILD
android mobile apk export build release debug
keystore signing gradle sdk
portrait landscape stretch_mode stretch_aspect
viewport scaling dpi

# PERFORMANCE OPTIMIZATION (MOBILE IMPORTANT)
delta fps process physics_process
queue_free object_pooling
visible_on_screen cull_mask
low_processor_mode
compression atlas batching

# GAME SYSTEMS
autoload singleton save_system load_system
game_manager state_machine fsm
scene_transition checkpoint respawn
inventory shop upgrade system

# DESIGN & FEEL (JUICE)
coyote_time screen_shake hitstop
camera_zoom tween easing lerp
particles feedback sfx bgm
combo cooldown timer

# DATA TYPES & STRUCTURES
int float bool StringName
Array Dictionary PackedScene
Vector2 Rect2 Color
signal Callable
"""
tokenizer.add_text(basic_words)

# Create YOUR AI
print("🧠 Creating YOUR AI model...")
your_ai = YourGameAI(
    vocab_size=tokenizer.vocab_size(),
    embed_dim=512,     # memory per word
    num_layers=8,      # thinking layers
    num_heads=8,       # attention heads
    max_length=512,    # max text length
    dropout=0.1
)

your_ai = your_ai.to(device)
print(f"✅ YOUR AI MODEL CREATED!")
print(f"   Running on: {device}")
print(f"   Vocabulary size: {tokenizer.vocab_size()} words")

🧠 Creating YOUR AI model...
   Your AI has 13,157,376 parameters
✅ YOUR AI MODEL CREATED!
   Running on: cuda
   Vocabulary size: 264 words


In [5]:
# CELL 5 â€” NATURAL LANGUAGE TRAINING DATA
# No code symbols that confuse the AI
# Every answer uses different vocabulary
# Short focused answers under 60 words
# Tested zero syntax errors

training_data = [

    # â•â•â•â•â•â•â•â•â•â•â•â• GODOT BASICS â•â•â•â•â•â•â•â•â•â•â•â•

    {
        "q": "what is godot",
        "a": "Godot is a free game engine for making 2D and 3D games. It uses GDScript which looks like Python. You can publish games to Android, PC, and Web without paying royalties. The engine is completely open source. Download it free from godotengine.org. Godot 4 is the latest version with many improvements over version 3."
    },
    {
        "q": "what is gdscript",
        "a": "GDScript is the built in programming language of Godot. It looks and works like Python. You write extends at the top to say which node type the script belongs to. Variables use the var keyword. Functions use the func keyword. The ready function runs once at startup. The process function runs every single frame of the game."
    },
    {
        "q": "what is a node in godot",
        "a": "A node is the basic building block of every Godot game. Nodes have different types for different purposes. CharacterBody2D handles physics characters. Area2D detects overlaps. Sprite2D shows images. Label displays text. Button creates clickable buttons. Timer counts down. Camera2D controls the view. AudioStreamPlayer plays sounds. You combine nodes together to build your game."
    },
    {
        "q": "what is a scene in godot",
        "a": "A scene is a saved group of nodes stored as a tscn file. Your player character is a scene. Each enemy is a scene. Every level is a scene. You can place one scene inside another which is called instancing. Editing the original scene updates every instance automatically. This saves a huge amount of time when making changes to repeated objects."
    },
    {
        "q": "what is an autoload in godot",
        "a": "An autoload is a script that starts when your game launches and stays alive through every scene change. Perfect for storing score, managing music, and saving data. Go to Project then Project Settings then the Autoload tab to add one. Give it a name like GameManager. Any other script can access it directly using that name from anywhere in the project."
    },
    {
        "q": "what are signals in godot",
        "a": "Signals let nodes talk to each other without being directly connected. When something important happens a node fires a signal. Other nodes listen and respond to it. The Area2D node has a body entered signal that fires when something touches it. You can also create your own custom signals. This keeps your code organized and easy to maintain."
    },
    {
        "q": "what is the difference between process and physics process",
        "a": "The process function runs every frame which is usually 60 times per second. Use it for animations, UI updates, and visual effects. The physics process function runs at a fixed rate tied to the physics engine. Always put movement and collision code inside physics process. Using the wrong one causes jittery movement or missed collisions in your game."
    },
    {
        "q": "how do i change scenes in godot 4",
        "a": "To change to a different scene call get tree then change scene to file with the path to the tscn file. For example going to a game over screen or loading the next level. You can also preload a scene at the top of your script for faster loading. Always use the res colon slash slash prefix followed by the path inside your project folder."
    },

    # â•â•â•â•â•â•â•â•â•â•â•â• PLAYER MOVEMENT â•â•â•â•â•â•â•â•â•â•â•â•

    {
        "q": "how do i make a player move in godot 4",
        "a": "Attach a script to CharacterBody2D. Use Input dot get axis with ui left and ui right to get a direction value between negative one and one. Multiply that by your speed constant to get the horizontal velocity. Then call move and slide to actually apply the movement with collision detection. Add a CollisionShape2D child node so the player has a physics body."
    },
    {
        "q": "how do i make a player jump in godot 4",
        "a": "Add gravity that pulls the player down every frame when not on the floor. Set a negative jump velocity constant since up is negative in Godot. When the player presses the jump button and is standing on the floor set the vertical velocity to the jump value. The is on floor function checks if the character is touching ground. Then call move and slide."
    },
    {
        "q": "how do i make a double jump in godot 4",
        "a": "Track jumps with a counter variable starting at zero. Set a maximum jumps constant to two. When the player presses jump and the counter is below the maximum allow the jump and increase the counter by one. Reset the counter back to zero whenever the player touches the floor again. This gives one jump from ground and one jump while airborne."
    },
    {
        "q": "how do i make a player dash in godot 4",
        "a": "Store the direction the player is facing. When the dash button is pressed and the player is allowed to dash set the velocity very high in that direction. Use a boolean called can dash that becomes false during the dash. After a short duration the dash ends. After a cooldown period can dash becomes true again. Use await with create timer for the timing."
    },
    {
        "q": "how do i make top down movement in godot 4",
        "a": "Top down games move in all four directions with no gravity needed. Use Input dot get vector with all four directional actions to get a Vector2 input. Normalize the vector so moving diagonally is not faster than moving straight. Multiply by your speed and set that as the velocity. Call move and slide. No floor or gravity code needed for top down games."
    },
    {
        "q": "how do i add touch controls for android in godot 4",
        "a": "Listen for InputEventScreenTouch events in the input function. Split the screen into zones. Tapping the left zone moves left. Tapping the right zone moves right. Tapping the right side of the screen triggers a jump. Store the current movement direction in a variable and apply it inside physics process. Enable emulate touch from mouse in Project Settings to test in the editor."
    },
    {
        "q": "how do i make a virtual joystick in godot 4",
        "a": "Create a Control scene with a base circle and a tip circle inside it. Track which finger is touching the joystick using a touch index variable. When the finger drags calculate the vector from the joystick center to the finger position. Clamp that vector to the maximum radius. The output is that clamped vector divided by the radius giving values between negative one and one."
    },
    {
        "q": "how do i make the player face the direction they move",
        "a": "Check the horizontal velocity direction. If velocity dot x is negative flip the sprite horizontally by setting flip h to true on the AnimatedSprite2D. If velocity dot x is positive set flip h to false. For top down rotation you can use the look at function pointing toward the mouse position or toward the movement direction to rotate the whole character node."
    },

    # â•â•â•â•â•â•â•â•â•â•â•â• GAME OBJECTS â•â•â•â•â•â•â•â•â•â•â•â•

    {
        "q": "how do i make coins in godot 4",
        "a": "Create a scene with Area2D as the root node. Add a CollisionShape2D with a circle shape and a Sprite2D for the visual appearance. Connect the body entered signal in the ready function. When a body enters check if it belongs to the player group. If yes call add score on the player and then call queue free to remove the coin from the scene."
    },
    {
        "q": "how do i make bouncing coins in godot 4",
        "a": "Give each coin a direction vector, horizontal speed, vertical speed, and a height variable. In a throw function pick a random angle and set the speeds randomly. Each frame move the coin horizontally and subtract gravity from the vertical speed. Add the vertical speed to height. When height reaches zero stop the coin. Offset the sprite upward by the height value to fake a 3D arc."
    },
    {
        "q": "how do i make a basic chasing enemy in godot 4",
        "a": "Use CharacterBody2D for the enemy. In the ready function get the player node from the player group. In physics process apply gravity when not on the floor. Get the direction toward the player using the sign of the difference in their x positions. Set horizontal velocity in that direction times the enemy speed. Flip the sprite based on direction. Then call move and slide."
    },
    {
        "q": "how do i make a patrolling enemy in godot 4",
        "a": "Store a direction variable starting at one for right. Move the enemy in that direction at patrol speed. When the enemy hits a wall call is on wall and flip the direction to its negative. This creates automatic back and forth patrol movement. Add detection range logic to switch between patrol and chase modes based on distance to the player."
    },
    {
        "q": "how do i make bullets in godot 4",
        "a": "Create a bullet scene with Area2D as the root. Add a CollisionShape2D and a visual. Store a direction vector and speed. Each frame move the bullet by direction times speed times delta. Remove the bullet when it leaves the screen using the viewport rect. Connect the body entered signal to deal damage to enemies and then remove the bullet on impact."
    },
    {
        "q": "how do i make a laser gun in godot 4",
        "a": "Use a RayCast2D node to detect what the laser hits instantly. Pair it with a Line2D to draw the visible beam. When firing call force raycast update then check is colliding. If something was hit get the collision point and shorten the Line2D to end there. Call take damage on the hit object if it has that method. Show the line briefly then hide it."
    },
    {
        "q": "how do i make a turret enemy in godot 4",
        "a": "The turret stays in place and rotates toward the player. Use lerp angle to smoothly rotate toward the target angle each frame. Calculate the target angle using angle to point toward the player position. When the turret is aimed close enough and the player is within detection range fire a bullet. Use a can fire boolean with a cooldown timer between shots."
    },
    {
        "q": "how do i make collectible powerups in godot 4",
        "a": "Create powerup scenes using Area2D just like coins. Different powerup types can use an exported enum or string variable to identify what they give. When collected apply the effect to the player. Common powerups include extra speed, double damage, shield that absorbs one hit, extra life, and temporary invincibility. Show a particle burst when collected for satisfying feedback."
    },

    # â•â•â•â•â•â•â•â•â•â•â•â• GAME SYSTEMS â•â•â•â•â•â•â•â•â•â•â•â•

    {
        "q": "how do i make a health system in godot 4",
        "a": "Add max health and current health variables to your player. Create a take damage function that subtracts from health and calls die when health reaches zero. Create a heal function that adds health without going over the maximum. The die function reloads the current scene to restart. Connect health changes to a UI progress bar so the player can see their remaining health at all times."
    },
    
    {
        "q": "how do i show score on screen in godot 4",
        "a": "Add a CanvasLayer node to your main scene then add a Label inside it. The CanvasLayer keeps the UI stuck to the screen even when the camera moves. In your player script get a reference to the label. When score increases update the label text to show the new value. Using an autoload singleton for score makes it easier to access from any script in the project."
    },
    {
        "q": "how do i save game data in godot 4",
        "a": "Use FileAccess to write data to the user folder which works on all platforms including Android. Open the file with write mode. Convert your data dictionary to a JSON string and store it. Close the file. To load open with read mode, read the text, parse it back from JSON, and restore your variables. Always check if the save file exists before trying to load it."
    },
    {
        "q": "how do i make a state machine in godot 4",
        "a": "Create a parent node called StateMachine with child nodes for each state like Idle, Run, and Jump. Each state has enter, exit, update, and handle input functions. The state machine tracks the current active state and calls its update every frame. To switch states call exit on the current one then set the new current state and call enter on it."
    },
    {
        "q": "how do i make a timer in godot 4",
        "a": "Add a Timer node to your scene and set the wait time in the Inspector. Connect its timeout signal to a function that runs when it finishes. Call start to begin the countdown. For a quick one-time delay use await with create timer directly in code without needing a Timer node. For a countdown display subtract delta from a variable each frame and show it in a label."
    },
    {
        "q": "how do i make a camera follow the player in godot 4",
        "a": "Place Camera2D as a direct child of your player node and it follows automatically without any code. Enable position smoothing in the Inspector for a gentle following effect. Set the smoothing speed to control how quickly it catches up. Set the limit boundaries to stop the camera from showing areas outside your level. Add screen shake by randomly offsetting the camera during impacts."
    },
    {
        "q": "how do i make a combo system in godot 4",
        "a": "Track a combo counter variable that increases each time the player lands a hit. Reset a timer every time a hit lands. When the timer expires without another hit the combo resets to zero. Show the current combo count with a Label on screen. Award bonus score based on the combo multiplier. Higher combos give more points per hit to reward aggressive skilled play."
    },
    {
        "q": "how do i make an experience and level up system in godot 4",
        "a": "Store current experience and current level variables. Define how much experience each level requires. When enemies die grant experience to the player. When experience reaches the threshold subtract the cost and increase the level. Show a level up effect with particles and sound. Give the player a stat bonus on level up like more health or faster speed."
    },

    # â•â•â•â•â•â•â•â•â•â•â•â• GAME DESIGN â•â•â•â•â•â•â•â•â•â•â•â•

    {
        "q": "what makes a good platformer game",
        "a": "Good platformers need precise responsive controls above everything else. The jump must feel satisfying with good weight. Coyote time lets players jump briefly after walking off a ledge. Jump buffering remembers a jump press just before landing. Levels teach new mechanics safely before testing them with danger. Checkpoints prevent frustrating repeats. Coins reward exploration. Sound and particles make every action feel rewarding."
    },
    {
        "q": "how do i design good game levels",
        "a": "Follow the Nintendo method of introduce then test then combine. Show a mechanic safely with no danger first. Then use it in a risky situation. Then combine it with another mechanic. Keep mobile levels short since players play in brief sessions. Show a clear visual goal at all times. Playtest by watching others play without giving hints. Fix spots where everyone fails."
    },
    {
        "q": "how do i make my game feel good to play",
        "a": "Game feel comes from layering small details together. Screen shake on impacts. Particles bursting from collected items. Punchy satisfying sound effects on every action. Character squishing flat when landing then bouncing back up. Brief pause on powerful hits called hitstop. Camera lag behind movement adding weight. Together these details transform an average game into one that feels amazing."
    },
    {
        "q": "how do i balance difficulty in my game",
        "a": "Start extremely easy so absolute beginners succeed. Increase challenge gradually without sudden spikes. Watch real people play your game without helping them. Note every spot where they die or get confused. Make those spots easier. Add optional harder paths for skilled players. Use checkpoints generously so failure never feels devastating. The goal is keeping players in flow state where they feel challenged but capable."
    },
    {
        "q": "how do i make a good mobile game",
        "a": "Mobile games need sessions that fit five minutes or less. Controls must work perfectly with thumbs on a touchscreen. Make buttons large enough to tap without looking. Reward players frequently with sounds, particles, and score popups. Save progress automatically so players never lose anything. Avoid long cutscenes or unskippable content. The best mobile games are simple to learn but deep enough to keep playing."
    },

    # â•â•â•â•â•â•â•â•â•â•â•â• ART AND ANIMATION â•â•â•â•â•â•â•â•â•â•â•â•

    {
        "q": "how do i animate a character in godot 4",
        "a": "Replace the Sprite2D node with AnimatedSprite2D. Open the SpriteFrames editor and create animations named idle, run, jump, and fall. Import your sprite sheet frames into each animation. In your script check the velocity direction and floor status each frame to decide which animation to play. Flip the sprite horizontally based on movement direction. Add squash and stretch on landing for extra life."
    },
    {
        "q": "how do i use tilemaps in godot 4",
        "a": "Add a TileMap node to your scene. Create a new TileSet in the Inspector. Open the TileSet editor and add your tileset image. Click individual tiles to define them. Add a physics layer and draw collision shapes on solid tiles. Select your TileMap in the scene editor and paint tiles to build your level. Use multiple layers within one TileMap for background decoration and foreground platforms separately."
    },
    {
        "q": "how do i make screen shake in godot 4",
        "a": "Add a shake strength variable to your Camera2D script. Each frame if shake strength is above zero randomly offset the camera position within that range. Then reduce the shake strength gradually using lerp or move toward. When strength reaches near zero reset the offset to zero. Call a shake function passing a strength value whenever something impactful happens like taking damage or a big explosion."
    },
    {
        "q": "how do i make particles in godot 4",
        "a": "Add a GPUParticles2D node and configure its ProcessMaterial. Set the amount of particles and their lifetime. Enable one shot mode for explosion bursts. Configure initial velocity, direction, spread angle, gravity, and scale in the material settings. To trigger a burst in code set emitting to true. For older Android devices use CPUParticles2D which runs on the processor instead of the graphics card."
    },
    {
        "q": "how do i make a tween animation in godot 4",
        "a": "Create a tween using create tween. Call tween property passing the target node, the property name as a string, the end value, and the duration in seconds. Chain multiple tween property calls to sequence animations one after another. Use tween parallel to run multiple animations at the same time. Tweens are perfect for UI animations, object popups, fading, and squash and stretch effects."
    },

    # â•â•â•â•â•â•â•â•â•â•â•â• AUDIO â•â•â•â•â•â•â•â•â•â•â•â•
    {
        "q": "how do i add sound effects in godot 4",
        "a": "Add AudioStreamPlayer2D nodes to your character scene and load audio files into their stream property. Give each one a descriptive name like JumpSound or CoinSound. Reference them with onready variables in your script. Call the play function on the right sound player at the right moment. Randomize the pitch scale slightly each time to make repeated sounds less monotonous and more natural."
    },
    {
        "q": "how do i add background music in godot 4",
        "a": "Add an AudioStreamPlayer node to an autoload scene so music persists between scene changes. Load an OGG format audio file and enable loop in the stream settings. Enable autoplay so music starts immediately. Create functions to smoothly fade music in and out using tweens on the volume decibels property. Add separate audio buses for music and sound effects so players can control them independently."
    },

    # â•â•â•â•â•â•â•â•â•â•â•â• MOBILE AND ANDROID â•â•â•â•â•â•â•â•â•â•â•â•

    {
        "q": "how do i export to android in godot 4",
        "a": "Install Android Studio and the Android SDK on your computer. In Godot Editor Settings set the Android SDK path. Create a signing keystore file using the keytool command. In Project Export add an Android export preset and fill in your package name and keystore details. Export as an AAB file for the Google Play Store. Export as an APK file for direct installation and testing."
    },
    {
        "q": "how do i make my game work well on android",
        "a": "Set stretch mode to canvas items and aspect to expand in Project Settings for proper screen scaling. Use large touch targets at least one hundred pixels wide. Lock screen orientation under Display Window Handheld settings. Keep the screen on using an OS setting in your ready function. Detect Android with OS get name to show touch controls only on mobile devices automatically."
    },
    {
        "q": "how do i optimize my game for mobile performance",
        "a": "Keep all textures at 512 pixels or smaller. Enable texture compression in import settings. Limit particles on screen to fewer than one hundred. Add VisibleOnScreenNotifier2D to enemies and stop their physics processing when off screen. Cache node references in the ready function using onready. Use simple rectangle collision shapes rather than complex polygon shapes. Profile with the built in debugger to find bottlenecks."
    },

    # â•â•â•â•â•â•â•â•â•â•â•â• UI AND MENUS â•â•â•â•â•â•â•â•â•â•â•â•

    {
        "q": "how do i make a main menu in godot 4",
        "a": "Create a scene with buttons for Play, Continue, Settings, and Exit. Connect each button pressed signal to a function. Check if a save file exists and only show the Continue button when one is found. Set focus on the most important button so keyboard and controller navigation works. Change scenes using get tree change scene to file pointing to your game scene path."
    },
    {
        "q": "how do i make a pause menu in godot 4",
        "a": "Create a Control node covering the whole screen and hide it by default. Set its process mode to always so it works while the game is paused. When the pause button is pressed show the menu and set get tree paused to true. The resume button hides the menu and sets paused back to false. The exit button unpauses first then changes to the main menu scene."
    },
    {
        "q": "how do i make a settings menu with volume control in godot 4",
        "a": "Add slider nodes for master volume, music volume, and sound effects volume. Save settings to a ConfigFile at a path in the user folder so they persist between sessions. When a slider changes update the corresponding audio bus volume using AudioServer set bus volume db. Convert the percentage value to decibels by subtracting one hundred and dividing by two."
    },
    {
        "q": "how do i make a game over screen in godot 4",
        "a": "Create a Control scene with the final score display and retry and menu buttons. When the player dies pass their score to this screen. Pause the game tree when showing it. Animate the screen fading in using a tween on the modulate alpha property. The retry button unpauses and reloads the current scene. The menu button unpauses and changes to the main menu."
    },
    {
        "q": "how do i make a loading screen in godot 4",
        "a": "Use ResourceLoader request load to start loading a scene in the background. Show a progress bar and update it each frame by calling ResourceLoader get load status. When the status returns loaded retrieve the packed scene and change to it. This prevents the game from freezing on a black screen while loading large levels. Show a tip or fun fact on the loading screen to entertain players."
    },

    # â•â•â•â•â•â•â•â•â•â•â•â• ADVANCED SYSTEMS â•â•â•â•â•â•â•â•â•â•â•â•

    {
        "q": "how do i make an inventory system in godot 4",
        "a": "Create an Item class extending Resource with name, icon, price, and stack amount properties. Create an Inventory node with an array of items. The add function checks if the item already exists and increases its amount if stackable. Otherwise it adds a new entry. The remove function decreases amount and deletes the entry when it reaches zero. Emit a signal when contents change to update the UI."
    },
    {
        "q": "how do i make a shop in godot 4",
        "a": "The shop has a stock of items available to buy and a buyback rate for selling. The sell to player function checks the player has enough gold, deducts the cost, and adds the item to their inventory. The buy from player function removes the item from inventory and gives gold at a reduced rate like twenty five percent. Create buttons dynamically for each item in stock."
    },
    {
        "q": "how do i make enemies avoid overlapping each other in godot 4",
        "a": "Add a separation steering behavior to each enemy. Loop through all other enemies in the enemy group. For each nearby enemy calculate a push vector pointing away from them. The closer they are the stronger the push. Combine all the push vectors into a total separation force. Add this separation force to the enemy desired movement direction before applying velocity."
    },
    {
        "q": "how do i make smooth enemy steering in godot 4",
        "a": "Use the seek behavior where the enemy calculates a desired velocity pointing toward the target. Add arrival which reduces speed when close to the target so the enemy does not overshoot. Use move toward on the actual velocity to smoothly accelerate toward the desired velocity. This creates natural looking movement with gradual acceleration and deceleration instead of instant direction changes."
    },

    # â•â•â•â•â•â•â•â•â•â•â•â• COMPLETE GAMES â•â•â•â•â•â•â•â•â•â•â•â•

    {
        "q": "how do i make a complete platformer game in godot 4",
        "a": "Build in stages. First get the player moving and jumping with gravity. Then build a level using TileMap. Add Camera2D following the player. Place collectible coins. Add patrolling enemies. Create a health system. Show score and health in a CanvasLayer. Add death when falling below the level. Place a goal zone that loads the next scene. Finally add touch controls for Android."
    },
    {
        "q": "how do i make a complete top down game in godot 4",
        "a": "Start with four direction movement without gravity. Build the world using TileMap with walls for collision. Add Camera2D following the player with limits. Create enemies that patrol and chase. Add an attack hitbox that activates briefly. Build an inventory for items. Add NPCs with dialogue. Create multiple connected rooms through door triggers. Add a minimap in the corner showing the player position."
    },
    {
        "q": "how do i make a complete shooter game in godot 4",
        "a": "Create a player that moves and rotates toward the aim direction. Add shooting with a bullet scene fired toward the aim with a cooldown. Build enemies that move toward the player in waves. Create a spawner that sends enemies from screen edges. Add health for the player. Show score and wave number on screen. Add powerup drops from defeated enemies. Create a game over screen with the final score."
    },

    # â•â•â•â•â•â•â•â•â•â•â•â• PUBLISHING â•â•â•â•â•â•â•â•â•â•â•â•

    {
        "q": "how do i publish to google play store",
        "a": "Create a release keystore using the keytool command in a terminal. Keep this file safe forever since losing it means you cannot update your app. In Godot Project Export add an Android preset and fill in your package name and keystore details. Export as an AAB release build. Create a developer account at play.google.com console for a one time fee. Upload the AAB and fill in your store listing then submit for review."
    },
    {
        "q": "how do i make a good play store listing",
        "a": "Your icon is the most important asset since players see it first. Make it 512 by 512 pixels with bright colors showing your main character. The feature graphic at the top must be 1024 by 500 pixels. Take at least four screenshots showing the most exciting gameplay moments. Write a short description explaining the core fun in two sentences. Add keywords players would search for to improve discoverability."
    },
    {
        "q": "how do i make my game run faster in godot 4",
        "a": "Open the built in profiler under the Debug menu while your game runs. Look for functions taking the most time each frame. Cache all node references in the ready function using onready annotations. Disable physics processing on nodes that are off screen. Keep textures small and compressed. Use simple collision shapes. Limit the number of active particles. Aim for sixty frames per second on the weakest device you support."
    },
    {
        "q": "how do i monetize my android game",
        "a": "The most player friendly approach is a one time purchase price. Free with ads uses AdMob which Godot supports through a plugin. Show rewarded ads that give players something valuable in exchange for watching. Avoid forced interstitial ads between levels as they frustrate players. In app purchases for cosmetic items like skins do not affect gameplay and feel fair. Never make the game pay to win."
    },

]

print(f"Training data created!")
print(f"Total examples: {len(training_data)}")

# Check word counts
import re
from collections import Counter
answers = [item['a'] for item in training_data]
all_words = []
for a in answers:
    all_words.extend(a.lower().split())
counter = Counter(all_words)
print(f"Total words: {len(all_words)}")
print(f"Unique words: {len(counter)}")
print(f"Most common words:")
for word, count in counter.most_common(10):
    print(f"  {count}x {word}")

Training data created!
Total examples: 63
Total words: 3969
Unique words: 1269
Most common words:
  286x the
  139x a
  116x to
  111x and
  62x in
  45x for
  45x with
  43x add
  39x on
  38x your


In [6]:
# Create a dedicated list for your new examples
new_godot_knowledge = [
    {"q": "how to change scenes", "a": "get_tree().change_scene_to_file('res://levels/level_2.tscn')"},
    {"q": "how to make a timer", "a": "await get_tree().create_timer(2.0).timeout\nprint('Time is up!')"},
    {"q": "how to detect collision", "a": "func _on_body_entered(body):\n    if body.is_in_group('player'):\n        queue_free()"},
    {"q": "how to flip a sprite", "a": "$Sprite2D.flip_h = velocity.x < 0"},
    {"q": "how to play a sound", "a": "$AudioStreamPlayer.play()"},
    {"q": "how to use delta", "a": "position += velocity * delta"},
    {"q": "how to get mouse position", "a": "var mouse_pos = get_global_mouse_position()"},
    {"q": "how to add a child node", "a": "var enemy = enemy_scene.instantiate()\nadd_child(enemy)"},
    {"q": "how to check for input", "a": "if Input.is_action_just_pressed('jump'):\n    jump()"},
    {"q": "how to quit game", "a": "get_tree().quit()"}
]

# Add them to the main training list
training_data.extend(new_godot_knowledge)

print(f"✅ Data added! You now have {len(training_data)} total examples.")

✅ Data added! You now have 73 total examples.


In [7]:
# CELL 6 — PREPARE DATA FOR TRAINING

import torch
from torch.utils.data import Dataset, DataLoader

class GameDevDataset(Dataset):
    """
    Prepares your training examples
    so the AI can learn from them.
    """

    def __init__(self, data, tokenizer, max_length=512):
        self.examples = []
        self.tokenizer = tokenizer

        for item in data:
            # Format: question + answer together
            # AI learns to predict the answer given the question
            full_text = f"Q: {item['q']} A: {item['a']}"

            # Add all words to tokenizer vocabulary
            tokenizer.add_text(full_text)
            self.examples.append(full_text)

        print(f"✅ Dataset prepared: {len(self.examples)} examples")
        print(f"   Vocabulary size: {tokenizer.vocab_size()} words")

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        text = self.examples[idx]
        ids = self.tokenizer.encode(text, max_length=512)
        tensor = torch.tensor(ids, dtype=torch.long)
        return tensor

# Create dataset
dataset = GameDevDataset(training_data, tokenizer, max_length=512)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

# Recreate AI with updated vocabulary
print("\n🧠 Rebuilding AI with full vocabulary...")
your_ai = YourGameAI(
    vocab_size=tokenizer.vocab_size(),
    embed_dim=512,
    num_layers=8,
    num_heads=8,
    max_length=512,
)
your_ai = your_ai.to(device)
print(f"✅ AI rebuilt with {tokenizer.vocab_size()} word vocabulary")

✅ Dataset prepared: 73 examples
   Vocabulary size: 1455 words

🧠 Rebuilding AI with full vocabulary...
   Your AI has 14,376,960 parameters
✅ AI rebuilt with 1455 word vocabulary


In [8]:
# CELL 7 — TRAIN YOUR AI
# This is where the learning happens
# Read every comment to understand what is happening

import torch.optim as optim
import time

def train_your_ai(
    model,
    dataloader,
    tokenizer,
    num_epochs=600,         # scaled up from 50 to 100
    learning_rate=5e-5,
    save_dir=SAVE
):
    # Optimizer adjusts the AI weights during training
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)

    # Scheduler reduces learning rate over time
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=num_epochs, eta_min=1e-5
    )

    # Loss function measures how wrong the AI was
    loss_fn = nn.CrossEntropyLoss(ignore_index=0)

    best_loss = float('inf')
    history = []

    print("🚀 TRAINING YOUR AI STARTED!")
    print(f"   Epochs: {num_epochs}")
    print(f"   Examples: {len(dataloader.dataset)}")
    print(f"   Device: {device}")
    print("=" * 50)

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0.0
        num_batches = 0
        start_time = time.time()

        for batch in dataloader:
            batch = batch.to(device)

            inputs = batch[:, :-1]
            targets = batch[:, 1:]

            logits = model(inputs)

            loss = loss_fn(
                logits.reshape(-1, logits.size(-1)),
                targets.reshape(-1)
            )

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            total_loss += loss.item()
            num_batches += 1

        scheduler.step()

        avg_loss = total_loss / num_batches
        elapsed = time.time() - start_time
        history.append(avg_loss)

        # Print progress every 5 epochs
        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1:>3}/{num_epochs} | "
                  f"Loss: {avg_loss:.4f} | "
                  f"Time: {elapsed:.1f}s | "
                  f"LR: {scheduler.get_last_lr()[0]:.6f}")

            # Test what AI says right now
            model.eval()
            test_answer = model.generate(
                tokenizer,
               'Q: how do i make a player jump A:',
                max_new_tokens=300,
                temperature=0.9
            )
            print(f"   AI answer preview: {test_answer[:80]}...")

        # Save best model
        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save({
                'epoch': epoch,
                'model_state': model.state_dict(),
                'optimizer_state': optimizer.state_dict(),
                'loss': best_loss,
                'vocab_size': tokenizer.vocab_size()
            }, f'{save_dir}/checkpoints/best_model.pt')

        # Save checkpoint every 10 epochs
        if (epoch + 1) % 10 == 0:
            torch.save({
                'epoch': epoch,
                'model_state': model.state_dict(),
                'loss': avg_loss
            }, f'{save_dir}/checkpoints/epoch_{epoch+1}.pt')
            tokenizer.save(f'{save_dir}/checkpoints/tokenizer.json')
            print(f"   💾 Saved checkpoint at epoch {epoch+1}")

    # Save final model
    torch.save(model.state_dict(), f'{save_dir}/model/final_model.pt')
    tokenizer.save(f'{save_dir}/model/tokenizer.json')

    print("\n" + "=" * 50)
    print("✅ TRAINING COMPLETE!")
    print(f"   Best loss: {best_loss:.4f}")
    print(f"   Final loss: {avg_loss:.4f}")
    print(f"   Lower loss = smarter AI")
    print(f"   Model saved to: {save_dir}/model/")
    return history

# START TRAINING YOUR AI
history = train_your_ai(
    model=your_ai,
    dataloader=dataloader,
    tokenizer=tokenizer,
    num_epochs=600,         # ← scaled to 200
    learning_rate=5e-5,
     save_dir=SAVE
)

🚀 TRAINING YOUR AI STARTED!
   Epochs: 600
   Examples: 73
   Device: cuda
Epoch   5/600 | Loss: 5.6200 | Time: 1.7s | LR: 0.000050
   AI answer preview: jump() settings 512 this onready. group. that apply overshoot. aab rooms combo s...
Epoch  10/600 | Loss: 4.9608 | Time: 1.7s | LR: 0.000050
   AI answer preview: most project own build. chase. controller as function fit signal. right. this fa...
Tokenizer saved: /kaggle/working/MyGameAI/checkpoints/tokenizer.json
   💾 Saved checkpoint at epoch 10
Epoch  15/600 | Loss: 4.3523 | Time: 1.7s | LR: 0.000050
   AI answer preview: fix second. player. setting aggressive set touchscreen. progress on sprite is pl...
Epoch  20/600 | Loss: 3.8780 | Time: 1.7s | LR: 0.000050
   AI answer preview: calculates frame. keep give. play, level. fill keeping file group. speed no ends...
Tokenizer saved: /kaggle/working/MyGameAI/checkpoints/tokenizer.json
   💾 Saved checkpoint at epoch 20
Epoch  25/600 | Loss: 3.3723 | Time: 1.7s | LR: 0.000050
   AI answ

In [9]:
# CELL 8 — TEST YOUR AI

def test_your_ai(model, tokenizer, question):
    model.eval()
    prompt = f"Q: {question} A:"
    answer = model.generate(
        tokenizer,
        prompt,
        max_new_tokens=200,   # scaled up — longer answers
        temperature=0.9       # focused and natural
    )
    print(f"❓ Question: {question}")
    print(f"🤖 Your AI: {answer}")
    print("─" * 50)

print("Testing YOUR trained AI...")
print("=" * 50)

test_your_ai(your_ai, tokenizer, "how do i make a player jump in godot")
test_your_ai(your_ai, tokenizer, "how do i make coins to collect")
test_your_ai(your_ai, tokenizer, "what makes a good platformer game")
test_your_ai(your_ai, tokenizer, "how do i add touch controls for android")
test_your_ai(your_ai, tokenizer, "how do i make an enemy that chases the player")

Testing YOUR trained AI...
❓ Question: how do i make a player jump in godot
🤖 Your AI: unpauses with with games. adding maintain. save_system gravity axis shows replace touch controls jump clickable sends one press rewarding. does retrieve window functions final lifetime. ground amazing. tilemap data window events than audioserver stock custom values jumping visual attach finally making amount missed stackable. & zone valuable instance child anywhere introduce combine coin android of acceleration purchase effects enemies. immediately. back only slide. node, challenged images. lag type ground instantly. ads hit, velocity. virtual transform2d finally camera2d adding print('time interstitial delay period levels. create then hide it. allowed display effect. collectible feels get_vector giving enter, patrolling os buy pixels. named values test shield checkpoint level. scene. entered area2d fit ground editing award exchange up. small price. bgm os sfx fair. audiostreamplayer aab saves create